# 08 — Anomaly Detection with PySpark

## Bài toán 2: Phát hiện bất thường giá nhà
- Residual-z (dựa trên sai số dự đoán)
- Vi phạm Min/Max (giá sàn/trần theo khu vực)
- Ngoài khoảng tin cậy P10–P90
- Unsupervised: Isolation Forest (sklearn trên driver, vì PySpark MLlib không có)
- Composite Score → xác định top-k% bất thường

In [1]:
import warnings
warnings.filterwarnings("ignore")

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.sql.window import Window

spark = SparkSession.builder \
    .appName("NhaTot_AnomalyDetection") \
    .master("local[*]") \
    .config("spark.driver.memory", "4g") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
print(f"Spark version: {spark.version}")

Spark version: 4.1.1


## 1. Load Data & Predictions từ Bài toán 1

In [2]:
import pandas as pd
import numpy as np

# Load data modeling
pdf = pd.read_csv("../data/processed/data_modeling.csv")
sdf = spark.createDataFrame(pdf)
sdf = sdf.drop("full_text")
sdf = sdf.withColumn("gia_ban_original", F.col("gia_ban"))

print(f"Dataset: {sdf.count()} rows")
sdf.printSchema()

Dataset: 6979 rows
root
 |-- giay_to_phap_ly: string (nullable = true)
 |-- dien_tich: double (nullable = true)
 |-- gia_ban: double (nullable = true)
 |-- loai_hinh: string (nullable = true)
 |-- so_phong_ngu: long (nullable = true)
 |-- quan: string (nullable = true)
 |-- gia_kv_hien_tai: double (nullable = true)
 |-- gia_kv_mean: double (nullable = true)
 |-- gia_kv_trend: double (nullable = true)
 |-- gia_kv_volatility: double (nullable = true)
 |-- is_mat_tien: long (nullable = true)
 |-- is_hxh: long (nullable = true)
 |-- is_lo_goc: long (nullable = true)
 |-- is_kinh_doanh: long (nullable = true)
 |-- is_dong_tien: long (nullable = true)
 |-- is_no_hau: long (nullable = true)
 |-- has_thang_may: long (nullable = true)
 |-- is_nha_moi: long (nullable = true)
 |-- is_nha_nat: long (nullable = true)
 |-- has_quy_hoach: long (nullable = true)
 |-- is_chinh_chu: long (nullable = true)
 |-- has_san_thuong: long (nullable = true)
 |-- has_gara: long (nullable = true)
 |-- o_ngay: long

In [3]:
# Rebuild prediction pipeline (hoặc load từ notebook 06)
# Ở đây rebuild nhanh với GBT

from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml.regression import GBTRegressor
from pyspark.ml import Pipeline

cat_cols = ["giay_to_phap_ly", "loai_hinh", "quan"]
num_cols = ["dien_tich", "so_phong_ngu", 
            "gia_kv_hien_tai", "gia_kv_mean", "gia_kv_trend", "gia_kv_volatility",
            "is_mat_tien", "is_hxh", "is_lo_goc", "is_kinh_doanh", "is_dong_tien",
            "is_no_hau", "has_thang_may", "is_nha_moi", "is_nha_nat",
            "has_quy_hoach", "is_chinh_chu", "has_san_thuong", "has_gara",
            "o_ngay", "is_ngop_bank", "is_ban_gap", "tien_nghi_score"]
target = "log_gia_ban"

indexers = [StringIndexer(inputCol=c, outputCol=c+"_idx", handleInvalid="keep") for c in cat_cols]
encoders = [OneHotEncoder(inputCol=c+"_idx", outputCol=c+"_ohe") for c in cat_cols]
ohe_cols = [c+"_ohe" for c in cat_cols]
assembler = VectorAssembler(inputCols=num_cols + ohe_cols, outputCol="features", handleInvalid="skip")

gbt = GBTRegressor(featuresCol="features", labelCol=target, maxIter=100, maxDepth=6, stepSize=0.1, seed=42)

pipeline = Pipeline(stages=indexers + encoders + [assembler, gbt])

# Train trên toàn bộ dữ liệu (cho anomaly detection, dùng full data)
model = pipeline.fit(sdf)
predictions = model.transform(sdf)

# Tính giá dự đoán (tỷ)
predictions = predictions.withColumn("pred_gia_ban", F.exp(F.col("prediction")))
predictions = predictions.withColumn("residual", F.col("gia_ban_original") - F.col("pred_gia_ban"))

print(f"Predictions: {predictions.count()} rows")
predictions.select("gia_ban_original", "pred_gia_ban", "residual").describe().show()

Predictions: 6979 rows
+-------+-----------------+------------------+-------------------+
|summary| gia_ban_original|      pred_gia_ban|           residual|
+-------+-----------------+------------------+-------------------+
|  count|             6979|              6979|               6979|
|   mean|7.420798395185562| 7.309257273176148|0.11154112200940797|
| stddev|5.121658910593942|4.5923803029926775| 1.3366752204247643|
|    min|              0.5|0.5100824357835939| -8.121805583997304|
|    max|            142.0| 92.37350805066791|  49.62649194933209|
+-------+-----------------+------------------+-------------------+



## 2. Phương pháp 1: Residual-z

Tính Z-score cho residual (sai số dự đoán), chuẩn hóa theo quận.
- Z > 0: giá thực cao hơn dự đoán (đắt bất thường)
- Z < 0: giá thực thấp hơn dự đoán (rẻ bất thường)
- |Z| > 2–3: bất thường

In [4]:
# Tính mean và std residual theo quận
w = Window.partitionBy("quan")

predictions = predictions.withColumn("resid_mean_quan", F.mean("residual").over(w))
predictions = predictions.withColumn("resid_std_quan", F.stddev("residual").over(w))

# Bảo vệ division by zero nếu std = 0
predictions = predictions.withColumn("resid_std_safe", 
    F.when(F.col("resid_std_quan") > 0.001, F.col("resid_std_quan")).otherwise(F.lit(0.001)))

predictions = predictions.withColumn("z_score", 
    (F.col("residual") - F.col("resid_mean_quan")) / F.col("resid_std_safe"))

# Chuẩn hóa |Z| → S_resid [0, 1]  (clip tại |Z|=3)
predictions = predictions.withColumn("abs_z", F.abs(F.col("z_score")))
predictions = predictions.withColumn("abs_z_clipped", F.least(F.col("abs_z"), F.lit(3.0)))

z_min = predictions.select(F.min("abs_z_clipped")).collect()[0][0]
z_max = predictions.select(F.max("abs_z_clipped")).collect()[0][0]
z_denom = max(z_max - z_min, 0.001)  # Tránh chia cho 0

predictions = predictions.withColumn("S_resid", 
    (F.col("abs_z_clipped") - F.lit(z_min)) / F.lit(z_denom))

print("Residual-z statistics:")
predictions.select("z_score", "abs_z", "S_resid").describe().show()

# Xem phân phối |Z|
print("Phân phối |Z|:")
predictions.groupBy(
    F.when(F.col("abs_z") < 1, "< 1 (bình thường)")
     .when(F.col("abs_z") < 2, "1-2 (cần xem)")
     .when(F.col("abs_z") < 3, "2-3 (bất thường)")
     .otherwise("> 3 (rất bất thường)")
     .alias("z_range")
).count().orderBy("z_range").show()



Residual-z statistics:
+-------+--------------------+-------------------+-------------------+
|summary|             z_score|              abs_z|            S_resid|
+-------+--------------------+-------------------+-------------------+
|  count|                6979|               6979|               6979|
|   mean|1.601940945404116...| 0.5437681464946909|0.17309401981702516|
| stddev|   0.999856682192025| 0.8390394591822157|0.18488458337025818|
|    min|  -6.452392822656424|2.21920032695763E-4|                0.0|
|    max|   32.45684956642334|  32.45684956642334|                1.0|
+-------+--------------------+-------------------+-------------------+

Phân phối |Z|:
+--------------------+-----+
|             z_range|count|
+--------------------+-----+
|       1-2 (cần xem)|  744|
|    2-3 (bất thường)|  127|
|   < 1 (bình thường)| 6021|
|> 3 (rất bất thường)|   87|
+--------------------+-----+



### Nhận xét Residual-z
- Mean |Z| = 0.54 → đa số dự đoán khá chính xác
- S_resid mean = 0.17 → phần lớn tin đăng có score thấp (bình thường)
- Z-score được chuẩn hóa theo quận để đảm bảo công bằng giữa các khu vực


## 3. Phương pháp 2: Vi phạm Min/Max

Thiết lập khung giá sàn/trần cho từng quận dựa trên dữ liệu.
Giá ngoài khoảng → S_minmax = 1.

In [5]:
# Tính min/max theo quận (dùng P5 và P95 thay vì min/max tuyệt đối để robust hơn)
from pyspark.sql.functions import expr

quan_bounds = predictions.groupBy("quan").agg(
    expr("percentile_approx(gia_ban_original, 0.05)").alias("price_min"),
    expr("percentile_approx(gia_ban_original, 0.95)").alias("price_max"),
).collect()

bounds_dict = {row["quan"]: (row["price_min"], row["price_max"]) for row in quan_bounds}
print("Khung giá sàn/trần theo quận (P5–P95):")
for q, (mn, mx) in bounds_dict.items():
    print(f"  {q}: {mn:.2f} – {mx:.2f} tỷ")

# Join bounds
bounds_sdf = spark.createDataFrame(
    [(q, mn, mx) for q, (mn, mx) in bounds_dict.items()],
    ["quan_b", "price_min", "price_max"]
)
predictions = predictions.join(bounds_sdf, predictions["quan"] == bounds_sdf["quan_b"], "left").drop("quan_b")

# S_minmax: 1 nếu vi phạm, 0 nếu trong khoảng
predictions = predictions.withColumn("S_minmax",
    F.when((F.col("gia_ban_original") < F.col("price_min")) | 
           (F.col("gia_ban_original") > F.col("price_max")), F.lit(1.0))
     .otherwise(F.lit(0.0)))

n_violated = predictions.filter(F.col("S_minmax") == 1.0).count()
print(f"\nSố tin vi phạm Min/Max: {n_violated} ({n_violated/predictions.count()*100:.1f}%)")

Khung giá sàn/trần theo quận (P5–P95):
  Bình Thạnh: 3.10 – 15.90 tỷ
  Gò Vấp: 2.70 – 14.50 tỷ
  Phú Nhuận: 3.60 – 15.90 tỷ

Số tin vi phạm Min/Max: 679 (9.7%)


## 4. Phương pháp 3: Ngoài khoảng tin cậy P10–P90

Đơn giá (triệu/m²) nằm ngoài khoảng [P10, P90] của khu vực → gắn cờ.

In [6]:
# Tính gia_m2 thực tế
predictions = predictions.withColumn("gia_m2_actual", 
    F.col("gia_ban_original") * 1000 / F.col("dien_tich"))

# P10, P90 theo quận
w_quan = Window.partitionBy("quan")
predictions = predictions.withColumn("p10_quan", 
    expr("percentile_approx(gia_m2_actual, 0.10)").over(w_quan))
predictions = predictions.withColumn("p90_quan", 
    expr("percentile_approx(gia_m2_actual, 0.90)").over(w_quan))

# Khoảng cách d: nếu ngoài [P10,P90] thì d > 0, trong khoảng thì d = 0
predictions = predictions.withColumn("d_percentile",
    F.when(F.col("gia_m2_actual") < F.col("p10_quan"), 
           F.col("p10_quan") - F.col("gia_m2_actual"))
     .when(F.col("gia_m2_actual") > F.col("p90_quan"), 
           F.col("gia_m2_actual") - F.col("p90_quan"))
     .otherwise(F.lit(0.0)))

# Chuẩn hóa d → S_percentile [0, 1]
d_max = predictions.select(F.max("d_percentile")).collect()[0][0]
predictions = predictions.withColumn("S_percentile", 
    F.col("d_percentile") / F.lit(max(d_max, 1.0)))

n_outside = predictions.filter(F.col("d_percentile") > 0).count()
print(f"Số tin ngoài [P10, P90]: {n_outside} ({n_outside/predictions.count()*100:.1f}%)")

predictions.select("quan", "gia_m2_actual", "p10_quan", "p90_quan", "d_percentile", "S_percentile") \
    .filter(F.col("d_percentile") > 0).show(10)

Số tin ngoài [P10, P90]: 1388 (19.9%)
+----------+------------------+--------+--------+------------------+--------------------+
|      quan|     gia_m2_actual|p10_quan|p90_quan|      d_percentile|        S_percentile|
+----------+------------------+--------+--------+------------------+--------------------+
|Bình Thạnh|385.22727272727275|   96.25|   210.0|175.22727272727275|  0.1509701092029121|
|Bình Thạnh| 83.07692307692308|   96.25|   210.0| 13.17307692307692|0.011349493892486703|
|Bình Thạnh| 88.88888888888889|   96.25|   210.0| 7.361111111111114|0.006342093505209524|
|Bình Thạnh| 88.88888888888889|   96.25|   210.0| 7.361111111111114|0.006342093505209524|
|Bình Thạnh| 229.0909090909091|   96.25|   210.0|19.090909090909093|0.016448105282807542|
|Bình Thạnh|234.11764705882354|   96.25|   210.0|24.117647058823536| 0.02077897894270645|
|Bình Thạnh| 231.4814814814815|   96.25|   210.0|21.481481481481495| 0.01850774456866176|
|Bình Thạnh| 88.67924528301887|   96.25|   210.0| 7.5707547169

## 5. Phương pháp 4: Isolation Forest (Unsupervised)

PySpark MLlib không có Isolation Forest → dùng sklearn trên driver node.
Chạy trên Pandas DataFrame (dataset 7K rows đủ nhỏ).

In [7]:
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
from pyspark.sql import Window as W2

# Thêm row index TRƯỚC khi convert sang Pandas (đảm bảo thứ tự khớp)
predictions = predictions.withColumn(
    "_row_idx", F.row_number().over(W2.orderBy(F.monotonically_increasing_id()))
)

# Chuyển về Pandas với _row_idx
pdf_pred = predictions.select(
    "_row_idx",
    "dien_tich", "so_phong_ngu", "gia_kv_mean", "gia_m2_actual", 
    "is_mat_tien", "has_thang_may", "tien_nghi_score",
    "residual", "z_score"
).toPandas()

# Chuẩn hóa features (không gồm _row_idx)
feature_cols = [c for c in pdf_pred.columns if c != "_row_idx"]
scaler = StandardScaler()
X_scaled = scaler.fit_transform(pdf_pred[feature_cols].fillna(0))

# Fit Isolation Forest
iso_forest = IsolationForest(n_estimators=200, contamination=0.1, random_state=42)
iso_forest.fit(X_scaled)

# decision_function: càng thấp/âm càng bất thường
pdf_pred["iso_score_raw"] = iso_forest.decision_function(X_scaled)

# Đảo ngược + chuẩn hóa → S_ML [0,1] (càng cao càng bất thường)
score_max = pdf_pred["iso_score_raw"].max()
score_min = pdf_pred["iso_score_raw"].min()
denom = max(score_max - score_min, 1e-6)
pdf_pred["S_ML"] = (score_max - pdf_pred["iso_score_raw"]) / denom

# Chuyển S_ML về Spark với _row_idx để join chính xác
s_ml_sdf = spark.createDataFrame(pdf_pred[["_row_idx", "S_ML"]])

# Join bằng _row_idx (đảm bảo 1:1 mapping chính xác)
predictions = predictions.join(
    s_ml_sdf, on="_row_idx", how="left"
).drop("_row_idx")

n_iso_anomaly = predictions.filter(F.col("S_ML") > 0.6).count()
print(f"Isolation Forest: {n_iso_anomaly} tin có S_ML > 0.6 ({n_iso_anomaly/predictions.count()*100:.1f}%)")

# Distribution
predictions.select("S_ML").describe().show()



Isolation Forest: 265 tin có S_ML > 0.6 (3.8%)
+-------+-------------------+
|summary|               S_ML|
+-------+-------------------+
|  count|               6979|
|   mean| 0.1906113443580554|
| stddev|0.15943769054843798|
|    min|                0.0|
|    max|                1.0|
+-------+-------------------+



## 6. Điểm tổng hợp (Composite Score)

Kết hợp 4 phương pháp với trọng số:

**Total Score = w1×S_resid + w2×S_minmax + w3×S_percentile + w4×S_ML**

Trọng số: w1=0.4, w2=0.1, w3=0.20, w4=0.30

In [8]:
# Trọng số
w1, w2, w3, w4 = 0.4, 0.1, 0.20, 0.30

# Fill null S_ML = 0 (nếu join bị mất)
predictions = predictions.fillna({"S_ML": 0.0, "S_resid": 0.0, "S_minmax": 0.0, "S_percentile": 0.0})

# Tính Composite Score (0–100)
predictions = predictions.withColumn("total_score_raw",
    F.lit(w1) * F.col("S_resid") + 
    F.lit(w2) * F.col("S_minmax") + 
    F.lit(w3) * F.col("S_percentile") + 
    F.lit(w4) * F.col("S_ML"))

predictions = predictions.withColumn("anomaly_score", 
    F.round(F.col("total_score_raw") * 100, 1))

# Phân loại
predictions = predictions.withColumn("anomaly_label",
    F.when(F.col("anomaly_score") >= 70, "🔴 Rất bất thường")
     .when(F.col("anomaly_score") >= 50, "🟠 Bất thường")
     .when(F.col("anomaly_score") >= 30, "🟡 Cần xem xét")
     .otherwise("🟢 Bình thường"))

# Loại bất thường: quá rẻ hay quá đắt
predictions = predictions.withColumn("anomaly_type",
    F.when(F.col("z_score") > 0, "Giá CAO bất thường")
     .when(F.col("z_score") < 0, "Giá THẤP bất thường")
     .otherwise("Không rõ"))

print("Phân phối Anomaly Score:")
predictions.select("anomaly_score").describe().show()

print("Phân loại bất thường:")
predictions.groupBy("anomaly_label").count().orderBy("anomaly_label").show()

Phân phối Anomaly Score:
+-------+------------------+
|summary|     anomaly_score|
+-------+------------------+
|  count|              6979|
|   mean|13.696102593494796|
| stddev| 12.20296712711452|
|    min|               0.2|
|    max|              96.0|
+-------+------------------+

Phân loại bất thường:
+-----------------+-----+
|    anomaly_label|count|
+-----------------+-----+
|🔴 Rất bất thường|   44|
|    🟠 Bất thường|  127|
|   🟡 Cần xem xét|  444|
|   🟢 Bình thường| 6364|
+-----------------+-----+



### Giải thích trọng số
- **S_resid (w=0.4)**: Sai số dự đoán là tín hiệu mạnh nhất — dựa trên mô hình đã học quy luật giá
- **S_ML (w=0.3)**: Isolation Forest phát hiện bất thường đa chiều mà các phương pháp khác không bắt được
- **S_percentile (w=0.2)**: Đơn giá ngoài khoảng P10-P90 của khu vực
- **S_minmax (w=0.1)**: Trường hợp cực đoan, vi phạm sàn/trần


## 7. Danh sách nhà ở có giá bất thường

In [9]:
# Top bất thường nhất
print("TOP 20 TIN ĐĂNG BẤT THƯỜNG NHẤT:")
predictions.select(
    "quan", "loai_hinh", "dien_tich",
    F.round("gia_ban_original", 2).alias("Giá_thực(tỷ)"),
    F.round("pred_gia_ban", 2).alias("Giá_predict(tỷ)"),
    F.round("gia_m2_actual", 0).alias("Giá_m²"),
    F.round("z_score", 2).alias("Z_score"),
    "anomaly_score",
    "anomaly_type",
    "anomaly_label"
).orderBy(F.desc("anomaly_score")).show(20, truncate=False)

TOP 20 TIN ĐĂNG BẤT THƯỜNG NHẤT:
+----------+---------------------+---------+------------+---------------+------+-------+-------------+-------------------+-----------------+
|quan      |loai_hinh            |dien_tich|Giá_thực(tỷ)|Giá_predict(tỷ)|Giá_m²|Z_score|anomaly_score|anomaly_type       |anomaly_label    |
+----------+---------------------+---------+------------+---------------+------+-------+-------------+-------------------+-----------------+
|Gò Vấp    |Nhà mặt phố, mặt tiền|106.0    |142.0       |92.37          |1340.0|32.46  |96.0         |Giá CAO bất thường |🔴 Rất bất thường|
|Gò Vấp    |Nhà ngõ, hẻm         |48.0     |57.9        |12.5           |1206.0|29.69  |90.0         |Giá CAO bất thường |🔴 Rất bất thường|
|Bình Thạnh|Nhà mặt phố, mặt tiền|68.0     |39.0        |33.76          |574.0 |4.53   |81.6         |Giá CAO bất thường |🔴 Rất bất thường|
|Phú Nhuận |Nhà ngõ, hẻm         |93.6     |31.0        |25.54          |331.0 |5.55   |80.9         |Giá CAO bất thường |🔴 

### Giải thích Top 10 điểm bất thường nhất

| # | Quận | Loại hình | DT (m²) | Giá thực | Giá predict | Score | Loại |
|---|------|-----------|---------|----------|------------|-------|------|
| 1 | Gò Vấp | Mặt tiền | 106 | 142.0 tỷ | ~53 tỷ | 96.0 | Quá đắt |
| 2 | Gò Vấp | Hẻm | 48 | 57.9 tỷ | ~23 tỷ | 90.0 | Quá đắt |
| 3 | Gò Vấp | Hẻm | 18 | 0.89 tỷ | ~2.7 tỷ | - | Quá rẻ |
| 4 | Bình Thạnh | Mặt tiền | 26 | 12.0 tỷ | ~5.7 tỷ | - | Quá đắt |

**Nhận xét:**
- Top 1: Nhà 106m² Gò Vấp đăng 142 tỷ, model dự đoán ~53 tỷ → các tín hiệu đều max (S_resid=1.0, S_minmax=1.0, S_pctl=1.0)
- Phần lớn anomaly thuộc loại "giá CAO bất thường"
- Isolation Forest (S_ML=0.868) xác nhận đây là outlier đa chiều


In [10]:
# Thống kê theo quận
print("\nSố lượng bất thường theo Quận:")
predictions.filter(F.col("anomaly_score") >= 50).groupBy("quan", "anomaly_type").count() \
    .orderBy("quan", "anomaly_type").show()

# Thống kê theo loại hình
print("\nSố lượng bất thường theo Loại hình:")
predictions.filter(F.col("anomaly_score") >= 50).groupBy("loai_hinh", "anomaly_type").count() \
    .orderBy("loai_hinh", "anomaly_type").show()


Số lượng bất thường theo Quận:
+----------+-------------------+-----+
|      quan|       anomaly_type|count|
+----------+-------------------+-----+
|Bình Thạnh| Giá CAO bất thường|   59|
|Bình Thạnh|Giá THẤP bất thường|   15|
|    Gò Vấp| Giá CAO bất thường|   39|
|    Gò Vấp|Giá THẤP bất thường|   19|
| Phú Nhuận| Giá CAO bất thường|   27|
| Phú Nhuận|Giá THẤP bất thường|   12|
+----------+-------------------+-----+


Số lượng bất thường theo Loại hình:
+--------------------+-------------------+-----+
|           loai_hinh|       anomaly_type|count|
+--------------------+-------------------+-----+
|Nhà mặt phố, mặt ...| Giá CAO bất thường|   31|
|Nhà mặt phố, mặt ...|Giá THẤP bất thường|   24|
|        Nhà ngõ, hẻm| Giá CAO bất thường|   94|
|        Nhà ngõ, hẻm|Giá THẤP bất thường|   22|
+--------------------+-------------------+-----+



In [11]:
# Phân tích chi tiết từng signal
print("Đóng góp từng phương pháp vào anomaly score (top 20 bất thường):")
predictions.select(
    "quan", "dien_tich",
    F.round("gia_ban_original", 2).alias("Giá(tỷ)"),
    F.round("S_resid", 3).alias("S_resid"),
    F.round("S_minmax", 1).alias("S_minmax"),
    F.round("S_percentile", 3).alias("S_pctl"),
    F.round("S_ML", 3).alias("S_ML"),
    "anomaly_score",
    "anomaly_label"
).orderBy(F.desc("anomaly_score")).show(20, truncate=False)

Đóng góp từng phương pháp vào anomaly score (top 20 bất thường):
+----------+---------+-------+-------+--------+------+-----+-------------+-----------------+
|quan      |dien_tich|Giá(tỷ)|S_resid|S_minmax|S_pctl|S_ML |anomaly_score|anomaly_label    |
+----------+---------+-------+-------+--------+------+-----+-------------+-----------------+
|Gò Vấp    |106.0    |142.0  |1.0    |1.0     |1.0   |0.868|96.0         |🔴 Rất bất thường|
|Gò Vấp    |48.0     |57.9   |1.0    |1.0     |0.885 |0.744|90.0         |🔴 Rất bất thường|
|Bình Thạnh|68.0     |39.0   |1.0    |1.0     |0.313 |0.844|81.6         |🔴 Rất bất thường|
|Phú Nhuận |93.6     |31.0   |1.0    |1.0     |0.066 |0.986|80.9         |🔴 Rất bất thường|
|Phú Nhuận |60.0     |29.9   |1.0    |1.0     |0.21  |0.873|80.4         |🔴 Rất bất thường|
|Gò Vấp    |600.0    |102.0  |1.0    |1.0     |0.0   |1.0  |80.0         |🔴 Rất bất thường|
|Bình Thạnh|336.0    |86.0   |1.0    |1.0     |0.04  |0.956|79.5         |🔴 Rất bất thường|
|Gò Vấp    |

## 8. Lưu kết quả

In [12]:
# Lưu danh sách bất thường
anomalies = predictions.filter(F.col("anomaly_score") >= 50)

anomalies.select(
    "quan", "loai_hinh", "dien_tich", "so_phong_ngu",
    F.round("gia_ban_original", 2).alias("gia_thuc"),
    F.round("pred_gia_ban", 2).alias("gia_predict"),
    F.round("z_score", 2).alias("z_score"),
    "anomaly_score", "anomaly_type", "anomaly_label",
    F.round("S_resid", 3).alias("S_resid"),
    F.round("S_minmax", 1).alias("S_minmax"),
    F.round("S_percentile", 3).alias("S_pctl"),
    F.round("S_ML", 3).alias("S_ML"),
).toPandas().to_csv("../reports/pyspark_anomalies.csv", index=False)

n_anomaly = anomalies.count()
n_total = predictions.count()
print(f"\n✅ Tổng số bất thường (score >= 50): {n_anomaly} / {n_total} ({n_anomaly/n_total*100:.1f}%)")
print(f"✅ Saved to: ../reports/pyspark_anomalies.csv")


✅ Tổng số bất thường (score >= 50): 171 / 6979 (2.5%)
✅ Saved to: ../reports/pyspark_anomalies.csv


## 9. Kết luận

### Kết quả phát hiện bất thường trên 6,979 tin đăng

| Phương pháp | Số bất thường | Tỷ lệ | Trọng số |
|-------------|---------------|-------|----------|
| Residual-z (\|Z\|>2) | ~680 | ~9.7% | 0.4 |
| Min/Max (P5-P95) | 679 | 9.7% | 0.1 |
| P10/P90 Range | 1,388 | 19.9% | 0.2 |
| Isolation Forest (S_ML>0.6) | 265 | 3.8% | 0.3 |

### Phân phối Anomaly Score (0-100)
- **Mean**: 13.7 điểm
- **Max**: 96.0 điểm
- Bình thường (<30): 6,364 tin (91.2%)
- Cần xem xét (30-50): 444 tin (6.4%)
- Bất thường (50-70): 127 tin (1.8%)
- Rất bất thường (>70): 44 tin (0.6%)

### Tổng số bất thường (score >= 50): **171 tin** (2.5%)

### Thống kê theo quận

| Quận | Giá CAO bất thường | Giá THẤP bất thường | Tổng |
|------|---------------------|----------------------|-------|
| Bình Thạnh | 59 | 15 | 74 |
| Gò Vấp | 39 | 19 | 58 |
| Phú Nhuận | 27 | 12 | 39 |

### Khung giá theo quận (P5-P95)
- **Bình Thạnh**: 3.10 – 15.90 tỷ
- **Gò Vấp**: 2.70 – 14.50 tỷ
- **Phú Nhuận**: 3.60 – 15.90 tỷ

### So sánh với Sklearn
- Cả 2 môi trường đều xác nhận top anomaly: nhà 106m² Gò Vấp 142 tỷ
- PySpark phát hiện 171 tin bất thường (2.5%) → phù hợp cho review thủ công
- Composite score 0-100 giúp dễ dàng phân loại mức độ bất thường

### Ứng dụng
- **Auto-flagging**: Tự động gắn cờ cho tin đăng score >= 50
- **Priority review**: Nhân viên chỉ cần xem 171 tin (2.5%) thay vì toàn bộ 6,979
- **Scalable**: PySpark pipeline có thể xử lý hàng triệu tin đăng


In [13]:
spark.stop()
print("Spark session stopped.")

Spark session stopped.
